# Preprocessing — Jalur **SVM + TF-IDF** (semua versi)

**Tujuan.** Memproses **seluruh 10.000 komentar berlabel** menjadi teks siap-SVM dan
menyimpannya ke koleksi **`processed_svm`**, lengkap dengan **flag 4 versi dataset**
(`in_set6k`, `in_balanced_set`, `in_set10k`, `in_balanced10k`).

**Langkah preprocessing (agresif — untuk bag-of-words / TF-IDF), berurutan:**
1. **Case folding** — seluruh teks dijadikan huruf kecil.
2. **Pembersihan (`clean_for_svm`)** — buang URL, mention `@`, hashtag `#`, emoji, angka, dan karakter non-alfabet.
3. **Normalisasi slang (`normalize_slang`)** — kata gaul→baku via `SLANG_DICT` (mis. `gk/tak`→`tidak`, `yg`→`yang`, `jkw`→`jokowi`); sebagian token derau (mis. `wkwk`, `bang`) dibuang.
4. **Tokenisasi** — pecah teks per spasi.
5. **Penghapusan stopword (`remove_stopwords`)** — buang kata fungsi umum & token ≤1 huruf. **Negasi `tidak/bukan/belum` sengaja DIPERTAHANKAN** agar makna sentimen ("tidak palsu") tidak hilang.
6. **Stemming (Sastrawi)** — potong imbuhan ke kata dasar (mis. `memalsukan`→`palsu`).

**Kenapa semua 10k sekaligus?** Pembersihan teks **tidak bergantung pada versi** — teks bersih
sebuah komentar sama saja di versi mana pun. Maka kita preprocess sekali, simpan flag versi, lalu
**pemilihan versi + split train/val/test dilakukan di notebook training** (`train_svm.ipynb`)
secara deterministik (`seed=42`, urut `comment_id`) — sehingga split tetap adil & identik dengan
IndoBERT untuk tiap versi. Pipeline identik dengan `src/text_normalizer.py`.

## 1. Dependency

Pasang pustaka yang dipakai notebook ini:
- **`pymongo[srv]` + `dnspython` + `certifi`** — koneksi ke MongoDB Atlas via skema `mongodb+srv://` (butuh resolusi DNS SRV) dengan TLS.
- **`python-dotenv`** — baca `MONGO_URI` dari file `.env` lokal.
- **`PySastrawi`** — *stemmer* Bahasa Indonesia (memotong imbuhan ke kata dasar).
- **`scikit-learn` / `pandas` / `numpy`** — manipulasi data & util.

Jalankan **sekali**. Di Colab, bila ada konflik versi setelah instal, *Runtime → Restart* lalu lanjut dari sel berikutnya.

In [1]:
# Pasang dependency (jalankan sekali; Sastrawi dipakai utk stemming)
%pip install -q "pymongo[srv]" dnspython certifi python-dotenv PySastrawi scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 2. Baca semua data berlabel (10k) + flag versi

Hubungkan ke Atlas lalu tarik **seluruh komentar berlabel** dari koleksi `raw_comments`
(filter `label` ada). `MONGO_URI` dicari berurutan: **env var → `.env` → input manual** (`getpass`),
jadi aman di lokal maupun Colab tanpa menaruh kredensial di notebook.

Kita ikut membawa **4 flag versi dataset** — `in_set6k` (v1 imbalanced 6k), `in_balanced_set`
(v2 balanced 3k), `in_set10k` (v3 imbalanced 10k), `in_balanced10k` (v4 balanced dari 10k);
flag yang `NaN` di-isi `False`. Keluaran: DataFrame `df` + ringkasan jumlah dokumen per versi.

In [2]:
# Koneksi Atlas + tarik semua komentar berlabel (10k) beserta flag versi
import os, pandas as pd
from pymongo import MongoClient
import certifi
MONGO_URI=os.environ.get("MONGO_URI","")
if not MONGO_URI:
    try:
        from dotenv import load_dotenv; load_dotenv(); MONGO_URI=os.environ.get("MONGO_URI","")
    except Exception: pass
if not MONGO_URI:
    from getpass import getpass; MONGO_URI=getpass("MONGO_URI: ")
DB=os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client=MongoClient(MONGO_URI,tlsCAFile=certifi.where(),serverSelectionTimeoutMS=20000)
client.admin.command("ping"); print("Koneksi MongoDB OK.")
LABELS=["Negatif","Netral","Positif"]
FLAGS=["in_set6k","in_balanced_set","in_set10k","in_balanced10k"]
proj={"_id":0,"comment_id":1,"video_id":1,"text":1,"label":1}
for fl in FLAGS: proj[fl]=1
# Ambil SEMUA komentar berlabel (10k) — preprocessing tak tergantung versi.
df=pd.DataFrame(list(client[DB]["raw_comments"].find({"label":{"$exists":True}},proj)))
for fl in FLAGS: df[fl]=df[fl].fillna(False)
print(f"{len(df)} dok berlabel (semua versi) dari Mongo")
print("Jumlah per versi:", {fl:int(df[fl].sum()) for fl in FLAGS})

Koneksi MongoDB OK.


10000 dok berlabel (semua versi) dari Mongo
Jumlah per versi: {'in_set6k': 6000, 'in_balanced_set': 3000, 'in_set10k': 10000, 'in_balanced10k': 5808}


## 3. Fungsi preprocessing (inline, identik dgn `src/text_normalizer.py`)

Definisi pipeline pembersihan SVM ditaruh **inline** (disalin dari `src/text_normalizer.py`)
supaya notebook bisa jalan mandiri di Colab **tanpa `import src`**. Isinya:
- **`SLANG_DICT`** — normalisasi kata gaul→baku (mis. `gk/tak`→`tidak`, `yg`→`yang`, `jkw`→`jokowi`); sebagian dipetakan ke string kosong (mis. `bang`, `wkwk`) untuk dibuang.
- **`STOPWORDS_ID`** — daftar stopword. **Negasi `tidak/bukan/belum` SENGAJA TIDAK dimasukkan** agar makna sentimen ("tidak palsu") tidak hilang.
- **Regex** pembersih URL/mention/hashtag/emoji/angka/non-alfabet.
- **`clean_for_svm` → `normalize_slang` → `tokenize` → `remove_stopwords`**, dirangkai oleh `preprocess_svm_python`.

Sel kedua memuat **Sastrawi stemmer**; `make_text_svm` = pipeline di atas **+ stemming** sebagai langkah terakhir.

In [3]:
# Pipeline pembersihan SVM (inline src/text_normalizer.py): clean -> slang -> stopword
import re
from typing import List

SLANG_DICT = {'aja': 'saja',
 'bagus': 'bagus',
 'baguss': 'bagus',
 'bang': '',
 'banget': 'sangat',
 'bener': 'benar',
 'bgt': 'sangat',
 'blm': 'belum',
 'blom': 'belum',
 'bngt': 'sangat',
 'bro': '',
 'buat': 'untuk',
 'bwt': 'untuk',
 'channel': 'saluran',
 'cuma': 'hanya',
 'dah': 'sudah',
 'deh': '',
 'dg': 'dengan',
 'dgn': 'dengan',
 'dong': '',
 'dr': 'dari',
 'dri': 'dari',
 'elu': 'kamu',
 'enggak': 'tidak',
 'g': 'tidak',
 'ga': 'tidak',
 'gak': 'tidak',
 'gakk': 'tidak',
 'gan': '',
 'gila': 'gila',
 'gilaa': 'gila',
 'gk': 'tidak',
 'gua': 'saya',
 'gue': 'saya',
 'gw': 'saya',
 'haha': '',
 'hehe': '',
 'hihi': '',
 'iya': 'ya',
 'iyaa': 'ya',
 'jelek': 'jelek',
 'jelekk': 'jelek',
 'jg': 'juga',
 'jga': 'juga',
 'jgn': 'jangan',
 'jkw': 'jokowi',
 'kak': '',
 'kalo': 'kalau',
 'kalu': 'kalau',
 'kan': '',
 'karna': 'karena',
 'keren': 'keren',
 'kerennn': 'keren',
 'kl': 'kalau',
 'klo': 'kalau',
 'klu': 'kalau',
 'kok': '',
 'komen': 'komentar',
 'konten': 'konten',
 'krn': 'karena',
 'kwkw': '',
 'lah': '',
 'lg': 'lagi',
 'lgi': 'lagi',
 'like': 'suka',
 'lo': 'kamu',
 'lu': 'kamu',
 'makasi': 'terima kasih',
 'makasih': 'terima kasih',
 'mantap': 'mantap',
 'mantep': 'mantap',
 'min': '',
 'mksh': 'terima kasih',
 'ngga': 'tidak',
 'nggak': 'tidak',
 'nih': '',
 'ntar': 'nanti',
 'ok': 'oke',
 'org': 'orang',
 'q': 'saya',
 'roi': 'roy',
 'sdh': 'sudah',
 'share': 'bagikan',
 'sih': '',
 'smgt': 'semangat',
 'sub': 'berlangganan',
 'subscribe': 'berlangganan',
 'sy': 'saya',
 'tak': 'tidak',
 'tar': 'nanti',
 'tau': 'tahu',
 'tdk': 'tidak',
 'tks': 'terima kasih',
 'tp': 'tapi',
 'tpi': 'tapi',
 'trs': 'terus',
 'trus': 'terus',
 'udah': 'sudah',
 'upload': 'unggah',
 'utk': 'untuk',
 'wkwk': '',
 'wkwkwk': '',
 'xD': '',
 'xd': '',
 'yg': 'yang',
 'yng': 'yang'}

STOPWORDS_ID = {'ada',
 'adalah',
 'agar',
 'akan',
 'amat',
 'anda',
 'antara',
 'apa',
 'atas',
 'atau',
 'bagaimana',
 'baru',
 'bawah',
 'beberapa',
 'besar',
 'bisa',
 'dalam',
 'dan',
 'dari',
 'dengan',
 'di',
 'dia',
 'dua',
 'ia',
 'ini',
 'itu',
 'jarang',
 'jika',
 'juga',
 'kadang',
 'kami',
 'kamu',
 'karena',
 'ke',
 'kecil',
 'ketika',
 'kita',
 'lagi',
 'lain',
 'lama',
 'lebih',
 'luar',
 'maka',
 'masih',
 'mengapa',
 'mereka',
 'namun',
 'nya',
 'oleh',
 'pada',
 'pernah',
 'pun',
 'saat',
 'saja',
 'sama',
 'sangat',
 'satu',
 'saya',
 'sebelum',
 'sedang',
 'sejak',
 'sekali',
 'selalu',
 'selama',
 'semua',
 'sering',
 'setelah',
 'setiap',
 'si',
 'sudah',
 'supaya',
 'tapi',
 'telah',
 'tetapi',
 'tiga',
 'untuk',
 'ya',
 'yang'}

_RE_URL = re.compile('https?://\\S+|www\\.\\S+', re.IGNORECASE)
_RE_MENTION = re.compile('@\\w+')
_RE_HASHTAG = re.compile('#\\w+')
_RE_EMOJI = re.compile('[😀-🙏🌀-🗿🚀-\U0001f6ff🜀-🝿🞀-\U0001f7ff🠀-\U0001f8ff🤀-🧿🨀-\U0001fa6f🩰-\U0001faff✂-➰Ⓜ-🉑🤦-🤷\u200d♀-♂☀-⭕⏏⏩⌚️〰]+', re.UNICODE)
_RE_NUMBER = re.compile('\\b\\d+\\b')
_RE_NON_ALPHA = re.compile('[^a-z\\s]')
_RE_MULTI_SPACE = re.compile('\\s+')

def clean_for_svm(text: str) -> str:
    """Cleaning agresif untuk jalur SVM + TF-IDF."""
    if not text or not isinstance(text, str):
        return ""
    t = text.lower()
    t = _RE_URL.sub(" ", t)
    t = _RE_MENTION.sub(" ", t)
    t = _RE_HASHTAG.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    t = _RE_NUMBER.sub(" ", t)
    t = _RE_NON_ALPHA.sub(" ", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    return t.strip()


def normalize_slang(text: str, slang_dict: dict = SLANG_DICT) -> str:
    """Ganti kata slang/informal dengan bentuk baku."""
    if not text:
        return ""
    tokens = text.split()
    normalized = [slang_dict.get(tok, tok) for tok in tokens]
    result = " ".join(tok for tok in normalized if tok)
    return _RE_MULTI_SPACE.sub(" ", result).strip()


def tokenize(text: str) -> List[str]:
    """Tokenisasi sederhana berdasarkan whitespace."""
    return text.split() if text else []


def remove_stopwords(tokens: List[str], stopwords: set = STOPWORDS_ID) -> List[str]:
    """Hapus stopword dari daftar token."""
    return [tok for tok in tokens if tok not in stopwords and len(tok) > 1]


def preprocess_svm_python(text: str) -> str:
    """Pipeline SVM tanpa stemming (stemming dilakukan terpisah via Sastrawi UDF)."""
    cleaned = clean_for_svm(text)
    normalized = normalize_slang(cleaned)
    tokens = tokenize(normalized)
    tokens = remove_stopwords(tokens)
    return " ".join(tokens)

In [4]:
# Stemmer Sastrawi; make_text_svm = pipeline di atas + stemming (langkah terakhir)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
_stemmer=StemmerFactory().create_stemmer()
def make_text_svm(text:str)->str:
    pre=preprocess_svm_python(text or ""); return _stemmer.stem(pre) if pre else pre

## 4. Transformasi langkah demi langkah

`trace_svm` membongkar **satu** contoh kalimat menjadi keluaran tiap tahap secara berurutan:
`clean_for_svm` → `normalize_slang` → `remove_stopword` → `stem (FINAL)`.
Tujuannya supaya terlihat **persis** bagaimana teks berubah di setiap langkah — berguna untuk
verifikasi pipeline sekaligus bahan penjelasan saat presentasi.

In [5]:
# Trace 1 kalimat: lihat hasil tiap tahap clean -> slang -> stopword -> stem
def trace_svm(t):
    s1=clean_for_svm(t); s2=normalize_slang(s1); s3=" ".join(remove_stopwords(tokenize(s2))); s4=_stemmer.stem(s3)
    print(f"RAW             : {t!r}"); print(f"1. clean_for_svm  : {s1!r}"); print(f"2. normalize_slang: {s2!r}")
    print(f"3. remove_stopword: {s3!r}"); print(f"4. stem (FINAL)   : {s4!r}")
trace_svm("Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti")

RAW             : 'Ijazahnya tidak palsu kok, jgn asal tuduh tanpa bukti'
1. clean_for_svm  : 'ijazahnya tidak palsu kok jgn asal tuduh tanpa bukti'
2. normalize_slang: 'ijazahnya tidak palsu jangan asal tuduh tanpa bukti'
3. remove_stopword: 'ijazahnya tidak palsu jangan asal tuduh tanpa bukti'
4. stem (FINAL)   : 'ijazah tidak palsu jangan asal tuduh tanpa bukti'


### Tabel before→after (sampel acak)

Ambil **6 komentar acak** (seed tetap `random_state=7` agar reproducible) lalu tampilkan
kolom `label`, **teks asli**, dan **hasil `svm`** berdampingan. Cara cepat menilai apakah
pembersihan agresif sudah tepat (tidak terlalu rakus membuang kata penting).

In [6]:
# Sampel 6 komentar acak (seed=7): bandingkan teks asli vs hasil SVM
import pandas as pd
pd.set_option("display.max_colwidth",55)
_s=df.sample(6,random_state=7).copy(); _s["svm"]=_s["text"].astype(str).map(make_text_svm)
_s[["label","text","svm"]]

,label,text,svm
1977,Negatif,Susah emang orang seperti itu..dulu koar koar ayo ...,susah emang orang seperti dulu koar koar ayo tunjuk...
3880,Negatif,kelompok orang2 sakit hati yg memburu ijajah pak jo...,kelompok orang sakit hati buru ijajah pak jokowi ka...
52,Positif,Kasihan ya.. PEMBICARANYA SEPERTI MENANGGUNG BEBAN ...,kasihan bicara seperti tanggung beban berat sehingg...
2551,Positif,JKW dirumah liat ini kejang2,jokowi rumah liat kejang
2246,Negatif,Kalau hati sudah busuk apapun yang kita lakukan tid...,kalau hati busuk apa laku tidak bakal rubah hati bu...
270,Positif,"Yg berbohong masalah ijazah jokowi, pasti segera di...",bohong masalah ijazah jokowi pasti segera laknat tu...


## 5. Terapkan ke seluruh 10k

Jalankan `make_text_svm` ke **seluruh 10k** komentar. Tambahkan `label_id`
(Negatif=0, Netral=1, Positif=2) dan **buang baris yang menjadi kosong** setelah preprocessing
(mis. komentar yang isinya hanya emoji/slang-yang-dibuang). Hasilnya `out_df`, siap disimpan.

In [7]:
# Terapkan make_text_svm ke seluruh 10k + label_id; buang baris kosong
LABEL2ID={l:i for i,l in enumerate(LABELS)}
out_df=df.copy(); out_df["label_id"]=out_df["label"].map(LABEL2ID)
out_df["svm"]=out_df["text"].astype(str).map(make_text_svm)
before=len(out_df); out_df=out_df[out_df["svm"].str.len()>0].reset_index(drop=True)
print(f"{before-len(out_df)} baris dibuang (kosong stlh preprocessing); sisa {len(out_df)}")

4 baris dibuang (kosong stlh preprocessing); sisa 9996


## 6. EDA & ringkasan kualitas

Cek kualitas hasil pembersihan: **distribusi kelas** (total 10k & per versi), **statistik panjang kata**
sebelum vs sesudah (mean/median/p95/max), **ukuran vocab unik**, jumlah teks tersisa **≤1 kata**,
dan jumlah **duplikat**. Indikator dini *over-cleaning* (teks terlalu pendek/kosong) atau bocoran duplikat.

In [8]:
# === EDA & ringkasan kualitas (seluruh 10k) ===
import numpy as np
print("Distribusi kelas (10k):"); print(out_df["label"].value_counts().reindex(LABELS).to_string())
wl_o=out_df["text"].astype(str).str.split().map(len); wl_p=out_df["svm"].astype(str).str.split().map(len)
print(f"\nPanjang(kata) ORIG: mean {wl_o.mean():.1f} median {wl_o.median():.0f} p95 {wl_o.quantile(.95):.0f} max {wl_o.max()}")
print(f"Panjang(kata) SVM : mean {wl_p.mean():.1f} median {wl_p.median():.0f} p95 {wl_p.quantile(.95):.0f} max {wl_p.max()}")
vocab=set(t for s in out_df["svm"].astype(str) for t in s.split())
print(f"\nVocab unik: {len(vocab)} | <=1 kata: {(wl_p<=1).sum()} | duplikat teks: {out_df['svm'].duplicated().sum()}")
print("\nDistribusi kelas per versi:")
for fl in FLAGS:
    sub=out_df[out_df[fl]]
    print(f"  {fl:<16} n={len(sub):<5} " + " ".join(f"{k}={v}" for k,v in sub['label'].value_counts().reindex(LABELS).items()))

Distribusi kelas (10k):
label
Negatif    4099
Netral     1934
Positif    3963

Panjang(kata) ORIG: mean 15.9 median 11 p95 42 max 689
Panjang(kata) SVM : mean 12.1 median 9 p95 31 max 540

Vocab unik: 12294 | <=1 kata: 23 | duplikat teks: 49

Distribusi kelas per versi:
  in_set6k         n=5998  Negatif=2621 Netral=1001 Positif=2376
  in_balanced_set  n=2998  Negatif=999 Netral=999 Positif=1000
  in_set10k        n=9996  Negatif=4099 Netral=1934 Positif=3963
  in_balanced10k   n=5804  Negatif=1935 Netral=1934 Positif=1935


## 7. Top term diskriminatif per kelas

Hitung term paling **membedakan** tiap kelas memakai skor **lift** = P(term|kelas) / P(term).
Term dengan lift tinggi muncul jauh lebih sering di satu kelas dibanding rata-rata global →
sinyal kuat yang nanti diberi bobot oleh TF-IDF. Hanya term berfrekuensi **≥8** yang dihitung
agar tidak bising oleh kata langka.

In [9]:
# === Top term diskriminatif per kelas (skor lift, seluruh 10k) ===
from collections import Counter
cnt={l:Counter() for l in LABELS}; tot=Counter(); nclass={}
for l in LABELS:
    sub=out_df[out_df["label"]==l]; nclass[l]=len(sub)
    for s in sub["svm"].astype(str):
        for t in set(s.split()): cnt[l][t]+=1; tot[t]+=1
N=len(out_df)
def top(l,k=12,minc=8):
    o=[(t,round((c/nclass[l])/(tot[t]/N),2)) for t,c in cnt[l].items() if tot[t]>=minc]
    return sorted(o,key=lambda x:-x[1])[:k]
for l in LABELS: print(f"[{l}] "+", ".join(f"{t}({v})" for t,v in top(l)))

[Negatif] dengki(2.44), siroy(2.44), pel(2.44), nusa(2.44), nusakambangan(2.44), iri(2.35), stress(2.34), tres(2.28), jeruji(2.25), soryo(2.24), jeblos(2.24), panci(2.19)
[Netral] banjir(4.13), pdi(3.88), sejahtera(3.45), ngaruh(3.23), kalaupun(3.23), capres(3.23), hibur(3.1), desa(3.1), saling(3.1), bencana(3.05), hotman(3.05), bosen(2.91)
[Positif] beban(2.52), sembunyi(2.52), istiqomah(2.52), raba(2.52), janggal(2.39), daftar(2.36), smp(2.33), pramuka(2.33), kaca(2.29), potonya(2.27), kendor(2.27), scan(2.27)


## 8. Tulis ke koleksi `processed_svm` (dengan flag versi, tanpa split)

Tulis 10k dokumen ke koleksi **`processed_svm`** (*upsert* berdasarkan `comment_id`) berisi
teks `svm`, `label`, `label_id`, dan **4 flag versi**. `delete_many({})` dijalankan dulu untuk
membersihkan isi lama. **Tidak ada field `split`** — pemilihan versi & pembagian train/val/test
dikerjakan deterministik di **`train_svm.ipynb`** (`seed=42`, urut `comment_id`) supaya identik dengan IndoBERT.

In [10]:
# Tulis 10k dok ke processed_svm (upsert, dgn flag versi, tanpa split)
from pymongo import UpdateOne
OUT=client[DB]["processed_svm"]; ops=[]
for r in out_df.to_dict("records"):
    doc={"comment_id":r["comment_id"],"video_id":r.get("video_id"),"text":r["text"],
         "svm":r["svm"],"label":r["label"],"label_id":int(r["label_id"])}
    for fl in FLAGS: doc[fl]=bool(r[fl])
    ops.append(UpdateOne({"comment_id":r["comment_id"]},{"$set":doc},upsert=True))
OUT.delete_many({})              # bersihkan isi lama (dulu cuma v2 + ada field split)
OUT.bulk_write(ops,ordered=False)
print("processed_svm:",OUT.count_documents({}),"dok (semua versi, tanpa split)")

processed_svm: 9996 dok (semua versi, tanpa split)


✅ **Selesai.** `processed_svm` berisi 10k teks siap-SVM + flag versi. Split & pemilihan versi dilakukan di **`train_svm.ipynb`**.